# LightGBM v4: evidence-gated GraphSAGE veto

CPU-friendly audit of frozen conditional routing. It scores no full TEST shards and reconstructs only 27 validation + 49 TEST veto candidates. Validation freezes the gate before TEST labels are opened.


In [ ]:
from pathlib import Path
import hashlib, json, sys, tarfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/evidence_veto_workspace')
WORK.mkdir(parents=True, exist_ok=True)

def unique_archive(name):
    matches = sorted(INPUT.glob('**/' + name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {name} under /kaggle/input; found {len(matches)}')
    return matches[0]

def extract_safe(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:*') as tar:
        for member in tar:
            target = (destination / member.name).resolve()
            if not target.is_relative_to(destination.resolve()) or not (member.isfile() or member.isdir()):
                raise ValueError(f'Unsafe archive member: {member.name}')
            if member.isfile():
                target.parent.mkdir(parents=True, exist_ok=True)
                with tar.extractfile(member) as src, target.open('wb') as dst:
                    while block := src.read(1 << 20):
                        dst.write(block)

def find_or_extract(marker, valid, archive_name, destination):
    # Kaggle datasets may already expose extracted archive contents. Prefer
    # those layouts, as the approved Host dataset does on the user's mount.
    extracted = sorted({p.parent for p in INPUT.glob('**/' + marker) if valid(p.parent)})
    if len(extracted) == 1:
        return extracted[0]
    if len(extracted) > 1:
        raise RuntimeError(f'Ambiguous extracted {marker}: {extracted}')
    archive = unique_archive(archive_name)
    extract_safe(archive, destination)
    if not valid(destination):
        raise ValueError(f'{archive_name} does not contain the expected {marker} layout')
    return destination

host = find_or_extract('manifest.json',
    lambda p: (p / 'assets/vocab.json').is_file() and (p / 'assets/split_manifest.json').is_file()
              and (p / 'volumes/ssd').is_dir(), 'host-input.tar', WORK / 'host')
bundle = find_or_extract('bundle_manifest.json',
    lambda p: (p / 'provenance/graph.sqlite').is_file()
              and (p / 'provenance/model_numpy.npz').is_file()
              and (p / 'code/cascade_runtime.py').is_file(),
    'provenance-cascade-bundle-v4.tar.gz', WORK / 'bundle')
extra = find_or_extract('package_manifest.json',
    lambda p: (p / 'previous/frozen_policy.json').is_file()
              and (p / 'cache/cache_manifest.json').is_file(),
    'v4-evidence-veto-input.tar.gz', WORK / 'evidence')
package = json.loads((extra / 'package_manifest.json').read_text())
if package['schema'] != 1:
    raise ValueError('Evidence input package schema mismatch')
for name, expected in package['files'].items():
    path = (extra / name).resolve()
    if not path.is_relative_to(extra.resolve()) or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f'Evidence input hash mismatch: {name}')
sys.path.insert(0, str(bundle / 'code'))
sys.path.insert(0, str(host))
print('Frozen inputs verified:', host, bundle, extra)


In [ ]:
runtime_source = '"""Audit evidence-gated high-ML veto using frozen conditional-v4 outputs.\n\nOnly the events already vetoed by the prior policy can change. Validation\nselects the evidence gate; TEST labels are used only after freezing that gate.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport sqlite3\nimport sys\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[2] / "preprocess"))\nfrom host_shards import HostShardDataset\nfrom cascade_runtime import metrics, safe, sha\nfrom ego import EgoGraphs\nfrom relational_sage import RelationalSAGE\n\nRULES = ("any_create", "root_create", "any_create_resolved",\n         "root_create_resolved", "any_create_untruncated")\nMIN_F1_GAIN = 0.0001\nMIN_DISTINCT_BENIGN_GRAPHS = 2\nMAX_ADDITIONAL_FN = 0\n\n\ndef atomic_json(path, value):\n    path = safe(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with tmp.open("w") as stream:\n        json.dump(value, stream, indent=2, sort_keys=True)\n        stream.flush(); os.fsync(stream.fileno())\n    os.replace(tmp, path)\n\n\ndef get_shard(data, shard):\n    from host_shards import decode_shard\n    return decode_shard(data.path_for(shard), data.width, shard)\n\n\ndef key_for_row(x, row):\n    start, stop = int(x.indptr[row]), int(x.indptr[row + 1])\n    cols = x.indices[start:stop]\n    if len(cols) > 1 and np.any(cols[:-1] >= cols[1:]):\n        raise ValueError("Host columns are not strictly increasing")\n    return cols.astype("<u4", copy=False).tobytes()\n\n\ndef evidence(graph):\n    creates = [(int(a), int(b)) for a, b, relation in graph["edges"]\n               if relation == "creates"]\n    flags = set(graph["flags"])\n    content = graph["x"].tobytes() + repr(graph["edges"]).encode()\n    return dict(nodes=int(len(graph["x"])), edges=len(graph["edges"]),\n                create_edges=len(creates), root_create=any(b == graph["root"] for _, b in creates),\n                unresolved="unresolved_relation" in flags,\n                truncated="truncated" in flags,\n                missing_identity="missing_identity_or_time" in flags,\n                graph_input_sha256=hashlib.sha256(content).hexdigest())\n\n\ndef gate(rule, item):\n    if item["missing_identity"]:\n        return False\n    create = item["create_edges"] > 0\n    root = item["root_create"]\n    if rule == "any_create":\n        return create\n    if rule == "root_create":\n        return root\n    if rule == "any_create_resolved":\n        return create and not item["unresolved"]\n    if rule == "root_create_resolved":\n        return root and not item["unresolved"]\n    if rule == "any_create_untruncated":\n        return create and not item["truncated"]\n    raise ValueError(f"Unknown gate: {rule}")\n\n\ndef revised_counts(conditional, rows, rule):\n    conf = dict(conditional)\n    kept_fp, kept_fn = [], []\n    reverted_fp, reverted_fn = [], []\n    for row in rows:\n        if gate(rule, row["evidence"]):\n            (kept_fp if row["label"] == 0 else kept_fn).append(row)\n            continue\n        if row["label"] == 0:\n            conf["fp"] += 1; conf["tn"] -= 1; reverted_fp.append(row)\n        else:\n            conf["tp"] += 1; conf["fn"] -= 1; reverted_fn.append(row)\n    return metrics(np.array([conf[k] for k in ("tp", "fp", "fn", "tn")], dtype=np.int64)), dict(\n        vetoed_fp=len(kept_fp), vetoed_fn=len(kept_fn),\n        reverted_fp=len(reverted_fp), rescued_fn=len(reverted_fn),\n        distinct_vetoed_benign_graphs=len({r["evidence"]["graph_input_sha256"] for r in kept_fp}),\n        distinct_vetoed_malicious_graphs=len({r["evidence"]["graph_input_sha256"] for r in kept_fn}),\n        vetoed_benign_sources=len({r["source_id"] for r in kept_fp}))\n\n\ndef select_rule(conditional, candidates):\n    base = metrics(np.array([conditional[k] for k in ("tp", "fp", "fn", "tn")], dtype=np.int64))\n    assessed = {}\n    for rule in RULES:\n        result, audit = revised_counts(conditional, candidates, rule)\n        eligible = (result["f1"] >= base["f1"] + MIN_F1_GAIN\n                    and result["confusion"]["fn"] <= conditional["fn"] + MAX_ADDITIONAL_FN\n                    and audit["distinct_vetoed_benign_graphs"] >= MIN_DISTINCT_BENIGN_GRAPHS)\n        assessed[rule] = dict(metrics=result, audit=audit, eligible=eligible)\n    winners = [r for r in RULES if assessed[r]["eligible"]]\n    winner = max(winners, key=lambda r: (assessed[r]["metrics"]["f1"],\n                 -assessed[r]["metrics"]["confusion"]["fn"], -RULES.index(r))) if winners else None\n    return dict(selected=winner or "conditional_unchanged", accepted=bool(winner),\n                rules=assessed, minimum_f1_gain=MIN_F1_GAIN,\n                minimum_distinct_benign_graphs=MIN_DISTINCT_BENIGN_GRAPHS,\n                maximum_additional_fn=MAX_ADDITIONAL_FN,\n                selection_basis="validation labels only; fixed prior graph threshold and scores")\n\n\ndef _rows(path):\n    with Path(path).open() as handle:\n        for line in handle:\n            if line.strip():\n                yield json.loads(line)\n\n\ndef _cache_graph(cache, split, index, shard, identity, n):\n    path = cache / f"{split}-graph-{index:04d}.npz"\n    receipt = json.loads(path.with_suffix(".json").read_text())\n    if (receipt["identity"] != identity or receipt["source_id"] != shard["source_id"]\n            or receipt["shard_sha256"] != shard["sha256"] or sha(path) != receipt["sha256"]):\n        raise ValueError(f"Frozen score cache mismatch: {path}")\n    with np.load(path, allow_pickle=False) as f:\n        scores = f["graph"]\n    if scores.shape != (n,):\n        raise ValueError("Score cache row count mismatch")\n    return scores\n\n\ndef validation_candidates(data, previous, cache, policy):\n    high = {}\n    for row in _rows(previous / "validation_per_vector_audit.jsonl"):\n        if row["validation_ml_band"] == "high":\n            high[bytes.fromhex(row["vector_key"])] = float(row["validation_ml_score"])\n    candidate = []\n    for index, shard in enumerate(data.meta["shards"]):\n        if shard["split"] != "val":\n            continue\n        x, y, lines = get_shard(data, shard)\n        scores = _cache_graph(cache, "val", index, shard, policy["identity"], len(y))\n        threshold = policy["policy"]["high_threshold"]\n        for i in range(len(y)):\n            key = key_for_row(x, i)\n            # Native Kaggle and NumPy cache differ at ~1e-17 for a threshold\n            # tie. The frozen native policy treats equality as positive.\n            if (key not in high or not np.isfinite(scores[i])\n                    or scores[i] >= threshold - 1e-15):\n                continue\n            candidate.append(dict(source_id=shard["source_id"], source_line=int(lines[i]),\n                label=int(y[i]), ml_score=high[key], graph_score=float(scores[i]),\n                vector_key=key.hex()))\n    labels = Counter(row["label"] for row in candidate)\n    if (labels[0], labels[1]) != (policy["policy"]["high_vetoed_fp"],\n                                   policy["policy"]["high_extra_fn"]):\n        raise ValueError(f"Validation veto candidates do not reproduce frozen policy: {dict(labels)}; "\n                         f"expected {policy[\'policy\'][\'high_vetoed_fp\']}/"\n                         f"{policy[\'policy\'][\'high_extra_fn\']}; malicious="\n                         f"{[(r[\'source_id\'][:8], r[\'source_line\'], r[\'graph_score\']) for r in candidate if r[\'label\']==1]}")\n    return candidate\n\n\ndef enrich(rows, db, ego, model, expected_split):\n    for row in rows:\n        target = db.execute("SELECT host,proc,t,split FROM target WHERE source=? AND line=?",\n                            (row["source_id"], row["source_line"])).fetchone()\n        if target is None or target[3] != expected_split:\n            raise ValueError("Frozen target/split mismatch")\n        graph = ego.build(*target[:3])\n        score = model.predict(graph)["probability"]\n        if abs(score - row["graph_score"]) > 1e-10:\n            raise ValueError("Cached GraphSAGE score differs from frozen checkpoint")\n        row["evidence"] = evidence(graph)\n    return rows\n\n\ndef run(manifest_path, bundle, previous, cache, output, volume_root_ssd=None):\n    manifest_path, bundle, previous, cache, output = map(lambda x: Path(x).resolve(),\n                                                         (manifest_path, bundle, previous, cache, output))\n    safe(output)\n    if output.exists() and any(output.iterdir()):\n        raise ValueError("Output must be a new empty directory")\n    for source in (bundle, previous, cache, manifest_path.parent):\n        if output == source or output.is_relative_to(source):\n            raise ValueError("Output overlaps an input")\n    policy = json.loads((previous / "frozen_policy.json").read_text())\n    identity = policy["identity"]\n    if not policy["policy"]["accepted"]:\n        raise ValueError("Prior conditional policy was not accepted")\n    # The approved seed02 manifest was relocated after the v4 run: its volume\n    # path changed, while run ID, split/vocabulary identities and shard hashes\n    # were preserved. Validate those contents below, not its historical bytes.\n    for path, expected in ((bundle / "bundle_manifest.json", identity["bundle_sha256"]),\n                           (bundle / "provenance/graph.sqlite", identity["graph_sha256"]),\n                           (bundle / "provenance/model_numpy.npz", identity["graph_model_sha256"]),\n                           (bundle / "models/lgbm_model.txt", identity["host_model_sha256"]),\n                           (manifest_path.parent / "assets/vocab.json", identity["schema_sha256"])):\n        if sha(path) != expected:\n            raise ValueError(f"Frozen input SHA mismatch: {path}")\n    cache_meta = json.loads((cache / "cache_manifest.json").read_text())\n    if cache_meta["identity"] != identity:\n        raise ValueError("Graph score cache identity mismatch")\n    data = HostShardDataset(manifest_path, volume_roots={"ssd": volume_root_ssd} if volume_root_ssd else None)\n    if (data.meta["vocab_size"] != identity["feature_width"]\n            or sha(manifest_path.parent / "assets/split_manifest.json") != data.meta["split_sha256"]\n            or sha(manifest_path.parent / "assets/vocab.json") != data.meta["vocabulary_sha256"]):\n        raise ValueError("Relocated manifest split/vocabulary identity mismatch")\n    old_val = json.loads((previous / "validation_metrics.json").read_text())\n    if old_val["identity"] != identity:\n        raise ValueError("Prior validation result identity mismatch")\n    val = validation_candidates(data, previous, cache, policy)\n    db_path = bundle / "provenance/graph.sqlite"\n    db = sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)\n    try:\n        db.execute("PRAGMA cache_size=-262144")\n        ego = EgoGraphs(db)\n        model = RelationalSAGE.load(bundle / "provenance/model_numpy.npz")\n        enrich(val, db, ego, model, 2)\n        choice = select_rule(old_val["selected"]["confusion"], val)\n        output.mkdir(parents=True, exist_ok=True)\n        atomic_json(output / "frozen_policy.json", dict(identity=identity, policy=choice,\n            prior_conditional_sha256=sha(previous / "frozen_policy.json")))\n        atomic_json(output / "validation_metrics.json", dict(identity=identity,\n            legacy=old_val["legacy"], conditional=old_val["selected"],\n            selected=choice["rules"][choice["selected"]]["metrics"] if choice["accepted"] else old_val["selected"],\n            candidate_count=len(val), gate=choice))\n        with (output / "validation_veto_evidence.jsonl").open("w") as f:\n            for row in val:\n                f.write(json.dumps(row, sort_keys=True) + "\\n")\n        frozen = json.loads((output / "frozen_policy.json").read_text())\n        if frozen["policy"] != choice:\n            raise ValueError("Frozen policy changed")\n        # Open TEST metrics and event labels only after the validation gate has\n        # been frozen on disk.\n        old_test = json.loads((previous / "test_metrics.json").read_text())\n        if old_test["identity"] != identity:\n            raise ValueError("Prior TEST result identity mismatch")\n        test = [r for r in _rows(previous / "event_transition_audit.jsonl")\n                if r["reason"] == "collision_high_veto"]\n        test_labels = Counter(r["label"] for r in test)\n        if (test_labels[0], test_labels[1]) != (43, 6):\n            raise ValueError("Prior TEST high-band transitions mismatch")\n        enrich(test, db, ego, model, 3)\n        rule = choice["selected"]\n        if choice["accepted"]:\n            selected, audit = revised_counts(old_test["selected"]["confusion"], test, rule)\n        else:\n            selected, audit = old_test["selected"], dict(vetoed_fp=43, vetoed_fn=6,\n                reverted_fp=0, rescued_fn=0)\n        atomic_json(output / "test_metrics.json", dict(identity=identity,\n            legacy=old_test["legacy"], conditional=old_test["selected"], selected=selected,\n            gate=rule, audit=audit, graph_calls_added=0,\n            prior_graph_calls=old_test["routing"]["selected_graph"],\n            evidence_graphs_reconstructed=len(val)+len(test)))\n        with (output / "test_veto_evidence.jsonl").open("w") as f:\n            for row in test:\n                row["gate_allowed"] = gate(rule, row["evidence"]) if choice["accepted"] else True\n                f.write(json.dumps(row, sort_keys=True) + "\\n")\n        prior_errors = list(_rows(previous / "test_errors_full_scores.jsonl"))\n        prior_transitions = list(_rows(previous / "event_transition_audit.jsonl"))\n        reverted = {(r["source_id"], r["source_line"]): r for r in test if not r["gate_allowed"]}\n        errors = {(r["source_id"], r["source_line"]): r for r in prior_errors}\n        for key, row in reverted.items():\n            if row["label"] == 0:\n                errors[key] = dict(source_id=key[0], source_line=key[1], label=0,\n                    ml_score=row["ml_score"], graph_score=row["graph_score"], collision=True,\n                    route_reason="evidence_gate_kept_lightgbm", prediction=1,\n                    legacy_prediction=1, error="FP")\n            else:\n                if key not in errors or errors[key]["error"] != "FN":\n                    raise ValueError("Reverted FN absent from prior error export")\n                del errors[key]\n        if len(errors) != selected["confusion"]["fp"] + selected["confusion"]["fn"]:\n            raise ValueError("Final errors do not reconcile")\n        with (output / "test_errors_full_scores.jsonl").open("w") as f:\n            for key in sorted(errors):\n                f.write(json.dumps(errors[key], sort_keys=True) + "\\n")\n        changes = [r for r in prior_transitions if (r["source_id"], r["source_line"]) not in reverted]\n        if len(changes) != 51 - len(reverted):\n            raise ValueError("Final transitions do not reconcile")\n        with (output / "event_transition_audit.jsonl").open("w") as f:\n            for row in changes:\n                f.write(json.dumps(row, sort_keys=True) + "\\n")\n        report = ["# Evidence-gated veto over frozen LightGBM v4 conditional routing", "",\n            "Validation selected the gate before TEST labels were read. The original Host/graph models, "\n            "scores, thresholds and graph construction remain unchanged.", "",\n            f"Selected gate: **{rule}**; accepted: **{choice[\'accepted\']}**.", "",\n            "| Split | Policy | TP | FP | FN | TN | F1 |", "|---|---|---:|---:|---:|---:|---:|"]\n        for split, doc in (("Validation", json.loads((output / "validation_metrics.json").read_text())),\n                           ("TEST", json.loads((output / "test_metrics.json").read_text()))):\n            for name in ("legacy", "conditional", "selected"):\n                m = doc[name]; c = m["confusion"]\n                report.append(f"| {split} | {name} | {c[\'tp\']} | {c[\'fp\']} | {c[\'fn\']} | {c[\'tn\']} | {m[\'f1\']:.6f} |")\n        report += ["", f"Only {len(val)} validation and {len(test)} TEST veto candidates needed graph reconstruction; "\n            "no full GraphSAGE rerun was performed.",\n            "The old TEST informed this research hypothesis. Its result is exploratory, not a fresh held-out confirmation. "\n            "Independent host/time/campaign validation is still needed."]\n        (output / "summary.md").write_text("\\n".join(report) + "\\n")\n        return dict(validation=choice, test=selected)\n    finally:\n        db.close()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    for name in ("manifest", "bundle", "previous", "cache", "output"):\n        parser.add_argument("--" + name, required=True)\n    parser.add_argument("--volume-root-ssd")\n    args = parser.parse_args()\n    run(args.manifest, args.bundle, args.previous, args.cache, args.output,\n        volume_root_ssd=args.volume_root_ssd)\n\n\nif __name__ == "__main__":\n    main()\n'
runtime_path = WORK / 'evidence_veto_runtime.py'
runtime_path.write_text(runtime_source)
sys.path.insert(0, str(WORK))
from evidence_veto_runtime import run


In [ ]:
output = Path('/kaggle/working/evidence_veto_v4_results')
result = run(host / 'manifest.json', bundle, extra / 'previous', extra / 'cache', output,
             volume_root_ssd=host / 'volumes/ssd')
print((output / 'summary.md').read_text())


In [ ]:
import zipfile
zip_path = Path('/kaggle/working/lgbm_v4_evidence_veto_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(output.iterdir()):
        if path.is_file():
            z.write(path, arcname=path.name)
print('Download:', zip_path)
